In [1]:
# 조건부 분기를 이용하여 질문 종류에 따라 다른 노드로 이동하기
# 수학/숫자 관련 질문 - math_node
# 일반 질문 -> chat_node

!pip install langgraph

from typing import TypedDict
from langgraph.graph import StateGraph, END, START
from IPython.display import Image, display

In [2]:
# State
class RouteState(TypedDict):
  query:str
  result:str

def math_node(state:RouteState) -> RouteState:
  q = state['query']
  answer = f"[MATH NODE] 수학/숫자 관련 질문으로 분류됨 : {q}"
  return {'query':q, "result":answer}

def chat_node(state:RouteState) -> RouteState:
  q = state['query']
  answer = f"[CHAT NODE] 일반 질문으로 분류됨 : {q}"
  return {'query':q, "result":answer}

def router_node(state:RouteState) -> RouteState:
  return state

# 분기 조건 함수
def route_dicision(state:RouteState) -> str:
  # 상태를 보고 다음으로 이동할 분기 키를 문자열로 반환
  q = state['query'].lower()

  if any(ch.isdigit() for ch in q) or any(word in q for word in ['더하기','빼기','곱하기','나누기']):
    return "math"
  else:
    return "chat"

In [6]:
# 그래프 구성
def build_graph():
  graph = StateGraph(RouteState)    # 그래프는 RouteState 구조의 데이터를 들고 다니는 파이프라인임

  graph.add_node("router", router_node)
  graph.add_node("math", math_node)
  graph.add_node("chat", chat_node)

  graph.set_entry_point("router")

  # 조건 분기
  graph.add_conditional_edges(
    "router",
    route_dicision,
    {
        "math": "math",
        "chat": "chat",
    },
  )

  graph.add_edge("math", END)
  graph.add_edge("chat", END)

  app = graph.compile()
  # 생성된 그래프 이미지 저장
  g = app.get_graph()
  png_bytes = g.draw_mermaid_png()

  with open("graph.png", 'wb') as f:
    f.write(png_bytes)

  return app

if __name__ == '__main__':
  app = build_graph()

  queries = [
      "2 더하기 3은 얼마야?",
      "장말철에 추천하는 차는 뭐가 있니?",
      "10 곱하기 20을 계산해 줘",
      "너를 소개해"
  ]

  for q in queries:
    print('질문 : ', q)
    final_state = app.invoke({'query':q, 'result':''})
    print('최종 답변 : ', final_state['result'])


질문 :  2 더하기 3은 얼마야?
최종 답변 :  [MATH NODE] 수학/숫자 관련 질문으로 분류됨 : 2 더하기 3은 얼마야?
질문 :  장말철에 추천하는 차는 뭐가 있니?
최종 답변 :  [CHAT NODE] 일반 질문으로 분류됨 : 장말철에 추천하는 차는 뭐가 있니?
질문 :  10 곱하기 20을 계산해 줘
최종 답변 :  [MATH NODE] 수학/숫자 관련 질문으로 분류됨 : 10 곱하기 20을 계산해 줘
질문 :  너를 소개해
최종 답변 :  [CHAT NODE] 일반 질문으로 분류됨 : 너를 소개해
